# Add FEMA Disaster-Declaration Exposure Variable

Pulls FEMA disaster declarations for Helene + Milton from the OpenFEMA API, codes each county on a **0–2 ordinal severity scale**, and merges with `pooled_dataset_100mi.csv`. Then refits the pooled OLS to test whether FEMA designation adds explanatory power over the existing predictors.

**Why FEMA:**
After we discovered the 50-mi cutoff was a sample-selection artefact and PRISM precipitation didn't quite clear p<0.05 (best aggregation: peak-day pop-weighted mean, p=0.092), we need an alternative exposure proxy. FEMA disaster declarations are an *integrated damage outcome* — they bundle wind + flooding + surge + infrastructure damage into a single ordinal measure that the government has vetted.

**Caveat (endogeneity):** FEMA is declared *after* damage assessment, so it's closer to an outcome than an exposure. The variable proxies for *intensity of integrated damage*, not pre-storm risk. We use it as a covariate to absorb storm-specific damage intensity that precipitation and track distance fail to capture, not as a causal exposure.

**Variable coding (per county):**
- `0` — no FEMA major-disaster declaration
- `1` — Public Assistance only (PA — infrastructure aid)
- `2` — Public Assistance + Individual Assistance (PA + IA — severe enough that households need direct federal aid)

**Aggregation to Helene clusters:** max across constituent counties (worst-affected county sets the cluster severity). Also report % counties with IA as a continuous alternative.

**Helene declarations:** DR-4827-NC, DR-4828-GA, DR-4829-SC, DR-4830-FL, DR-4831-TN, DR-4832-VA
**Milton declaration:** DR-4834-FL

**Outputs:**
- `results/exposure/fema_county_declarations.csv` — raw OpenFEMA pull
- `results/exposure/county_fema_severity.csv` — 0–2 ordinal score per county per hurricane
- `results/local_level/regression/pooled_dataset_100mi_with_fema.csv` — pooled augmented
- `results/exposure/ols_fema_comparison.csv` — coefficient comparison across model specs

In [ ]:
import os, time, warnings
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

BASE = Path("..")
EXPOSURE_DIR = BASE / "results/exposure"
EXPOSURE_DIR.mkdir(parents=True, exist_ok=True)

## §1. Pull FEMA declarations from OpenFEMA API

In [ ]:
# Disaster numbers for Helene (multiple states) and Milton
HELENE_DISASTERS = [4827, 4828, 4829, 4830, 4831, 4832]  # NC, GA, SC, FL, TN, VA
MILTON_DISASTERS = [4834]                                  # FL
DISASTERS = HELENE_DISASTERS + MILTON_DISASTERS
HURRICANE_OF = {n: "helene" for n in HELENE_DISASTERS}
HURRICANE_OF.update({n: "milton" for n in MILTON_DISASTERS})

# OpenFEMA v2 API
BASE_URL = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"

def fetch_disaster(disaster_num, top=1000, skip=0):
    params = {
        "$filter": f"disasterNumber eq {disaster_num}",
        "$top": top, "$skip": skip,
    }
    r = requests.get(BASE_URL, params=params, timeout=60,
                     headers={"User-Agent": "research-script"})
    r.raise_for_status()
    return r.json().get("DisasterDeclarationsSummaries", [])

rows = []
for dnum in DISASTERS:
    print(f"  DR-{dnum} ({HURRICANE_OF[dnum]}):", end=" ")
    recs = fetch_disaster(dnum)
    print(f"{len(recs)} county-rows")
    rows.extend(recs)
    time.sleep(0.5)

fema_raw = pd.DataFrame(rows)
print(f"\nTotal FEMA county-disaster rows: {len(fema_raw)}")
print("Sample columns:")
print(fema_raw.columns.tolist())
fema_raw.to_csv(EXPOSURE_DIR / "fema_county_declarations.csv", index=False)
print(f"Saved: {EXPOSURE_DIR / 'fema_county_declarations.csv'}")

## §2. Build the 0–2 ordinal severity score per county per hurricane

In [ ]:
# Key columns from OpenFEMA DisasterDeclarationsSummaries:
# - fipsStateCode, fipsCountyCode → assemble 5-digit GEOID
# - ihProgramDeclared, iaProgramDeclared, paProgramDeclared, hmProgramDeclared (booleans)
# - disasterNumber
# - placeCode = 6-digit county FIPS (alternative)

# Some declarations roll up at the state level (placeCode = '0' or NaN). Keep only county-level rows.
fema = fema_raw.copy()
fema["hurricane"] = fema["disasterNumber"].map(HURRICANE_OF)

# Build GEOID: state FIPS (2-digit) + county FIPS (3-digit)
fema["state_fips"] = fema["fipsStateCode"].astype(str).str.zfill(2)
fema["county_fips"] = fema["fipsCountyCode"].astype(str).str.zfill(3)
fema["GEOID"] = (fema["state_fips"] + fema["county_fips"]).astype(int)

# Filter to county-level (county_fips != '000') AND PA or IA designated (skip pure HM-only rows)
fema_county = fema[(fema["county_fips"] != "000") &
                    (fema[["paProgramDeclared", "iaProgramDeclared"]].any(axis=1))].copy()
print(f"County-level rows with PA or IA: {len(fema_county)}")

# For each (GEOID, hurricane) pair, compute the max severity across multiple declarations
severity = fema_county.groupby(["GEOID", "hurricane"]).agg(
    pa_declared=("paProgramDeclared", "max"),
    ia_declared=("iaProgramDeclared", "max"),
).reset_index()
severity["fema_severity"] = severity["pa_declared"].astype(int) + severity["ia_declared"].astype(int)
# fema_severity = 0 (neither), 1 (PA only), 2 (PA+IA). Note: IA without PA is rare.

print(f"\nSeverity distribution by hurricane:")
print(severity.groupby("hurricane")["fema_severity"].value_counts().unstack(fill_value=0))

severity.to_csv(EXPOSURE_DIR / "county_fema_severity.csv", index=False)
print(f"\nSaved: {EXPOSURE_DIR / 'county_fema_severity.csv'}")

## §3. Spot-check known counties

In [ ]:
# Known high-damage Helene counties — these SHOULD have IA (severity=2)
expected_helene_ia = [
    ("Buncombe NC", 37021),
    ("Yancey NC",   37199),
    ("Mitchell NC", 37121),
    ("Avery NC",    37011),
    ("Watauga NC",  37189),
    ("Haywood NC",  37087),
    ("Henderson NC", 37089),
    ("Madison NC",  37115),
    ("Taylor FL",   12123),  # Helene landfall
]
# Known high-damage Milton counties — coastal FL
expected_milton_ia = [
    ("Pinellas FL",     12103),
    ("Hillsborough FL", 12057),
    ("Sarasota FL",     12115),
    ("Manatee FL",      12081),
    ("Charlotte FL",    12015),
    ("Lee FL",          12071),
    ("Volusia FL",      12127),
]

print("Helene high-damage spot-check:")
for name, geoid in expected_helene_ia:
    s = severity[(severity["GEOID"] == geoid) & (severity["hurricane"] == "helene")]
    val = s["fema_severity"].iloc[0] if len(s) else "NOT FOUND"
    print(f"  {name:<20} GEOID={geoid}  severity={val}")

print("\nMilton high-damage spot-check:")
for name, geoid in expected_milton_ia:
    s = severity[(severity["GEOID"] == geoid) & (severity["hurricane"] == "milton")]
    val = s["fema_severity"].iloc[0] if len(s) else "NOT FOUND"
    print(f"  {name:<20} GEOID={geoid}  severity={val}")

## §4. Aggregate to Helene clusters

Two aggregations:
- **`fema_severity_max`** — worst-affected county sets the cluster level (ordinal 0–2)
- **`fema_pct_ia`** — fraction of cluster counties with IA designation (continuous 0–1)

In [ ]:
# Helene 100mi cluster assignments
hel_assign = pd.read_csv("../results/local_level/helene_100mi/county_cluster_assignments.csv")
hel_assign["GEOID"] = hel_assign["GEOID"].astype(int)

# Inner join with severity (Helene only); counties not in `severity` get severity=0
hel_sev = severity[severity["hurricane"] == "helene"][["GEOID", "fema_severity", "ia_declared"]]
hel_full = hel_assign.merge(hel_sev, on="GEOID", how="left")
hel_full["fema_severity"] = hel_full["fema_severity"].fillna(0).astype(int)
hel_full["ia_declared"] = hel_full["ia_declared"].fillna(False)

# Cluster aggregation
hel_cluster_fema = hel_full.groupby("cluster").agg(
    fema_severity_max=("fema_severity", "max"),
    fema_severity_mean=("fema_severity", "mean"),
    fema_pct_ia=("ia_declared", "mean"),
    n_counties=("GEOID", "count"),
).reset_index()
print(f"Helene 100mi clusters with FEMA aggregation: N={len(hel_cluster_fema)}")
print("\nSeverity distribution at cluster level (max):")
print(hel_cluster_fema["fema_severity_max"].value_counts().sort_index())
print("\nSample clusters (sorted by max severity):")
print(hel_cluster_fema.sort_values("fema_severity_max", ascending=False).head(10).to_string(index=False))

## §5. Merge into pooled_dataset_100mi

In [ ]:
pooled = pd.read_csv("../results/local_level/regression/pooled_dataset_100mi.csv")
print(f"Loaded pooled_100mi: N={len(pooled)}")

# Milton: NAME → county name → GEOID via county_metadata
mil_meta = pd.read_csv("../results/local_level/milton_100mi/county_metadata.csv")
mil_meta["GEOID"] = mil_meta["GEOID"].astype(int)
mil_sev = severity[severity["hurricane"] == "milton"][["GEOID", "fema_severity", "ia_declared"]]
mil_rows = pooled[pooled["hurricane"] == "milton"].merge(
    mil_meta[["GEOID", "NAME"]], on="NAME", how="left")
mil_with = mil_rows.merge(mil_sev, on="GEOID", how="left")
mil_with["fema_severity"] = mil_with["fema_severity"].fillna(0).astype(int)
mil_with["ia_declared"] = mil_with["ia_declared"].fillna(False)
mil_with["fema_severity_max"] = mil_with["fema_severity"]   # 1 county = max is itself
mil_with["fema_severity_mean"] = mil_with["fema_severity"]
mil_with["fema_pct_ia"] = mil_with["ia_declared"].astype(float)
mil_with = mil_with.drop(columns=["GEOID", "fema_severity", "ia_declared"])

# Helene: NAME → cluster id
hel_rows = pooled[pooled["hurricane"] == "helene"].copy()
hel_rows["cluster"] = hel_rows["NAME"].str.replace("Cluster_", "").astype(int)
hel_with = hel_rows.merge(
    hel_cluster_fema[["cluster", "fema_severity_max", "fema_severity_mean", "fema_pct_ia"]],
    on="cluster", how="left").drop(columns=["cluster"])

aug = pd.concat([mil_with, hel_with], ignore_index=True)
out_path = "../results/local_level/regression/pooled_dataset_100mi_with_fema.csv"
aug.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Augmented N={len(aug)}")
print(f"Missing FEMA values: {aug[['fema_severity_max','fema_pct_ia']].isna().sum().to_dict()}")
print("\nCross-tab: severity_max × hurricane")
print(pd.crosstab(aug["fema_severity_max"], aug["hurricane"]))

## §6. Does FEMA add explanatory power?

Refit the pooled OLS at 100mi without FEMA, then with each FEMA aggregation, and watch `dist_to_track_mi`, `is_milton`, and Adj. R².

In [ ]:
BASE_FEATURES = [
    "median_household_income", "pct_no_vehicle", "pct_white", "nchs_code",
    "total_population", "dist_to_track_mi", "insurance_coverage_pct",
    "is_milton", "is_coastal",
]

def fit_for_dv(df, features, dv):
    d = df.dropna(subset=features + [dv]).reset_index(drop=True)
    Xz = pd.DataFrame(StandardScaler().fit_transform(d[features]),
                       columns=features, index=d.index)
    return sm.OLS(d[dv], sm.add_constant(Xz)).fit(), len(d)

DVs = ["largest_drop_within", "recovery_days_within", "largest_drop_inflow"]
FEMA_VARS = ["fema_severity_max", "fema_severity_mean", "fema_pct_ia"]

summary_rows = []
for dv in DVs:
    print("\n" + "=" * 88)
    print(f"DV: {dv}")
    print("=" * 88)
    m0, n0 = fit_for_dv(aug, BASE_FEATURES, dv)
    print(f"  baseline (no FEMA):  N={n0}  Adj.R²={m0.rsquared_adj:.3f}  "
          f"dist_to_track β={m0.params['dist_to_track_mi']:+.3f} (p={m0.pvalues['dist_to_track_mi']:.4f})")
    summary_rows.append({"dv": dv, "spec": "baseline", "N": n0,
                          "adjR2": round(m0.rsquared_adj, 3),
                          "beta_dist": round(m0.params["dist_to_track_mi"], 3),
                          "p_dist": round(m0.pvalues["dist_to_track_mi"], 4),
                          "beta_fema": None, "p_fema": None})
    for fema_var in FEMA_VARS:
        m, n = fit_for_dv(aug, BASE_FEATURES + [fema_var], dv)
        bf, pf = m.params[fema_var], m.pvalues[fema_var]
        bd, pd_ = m.params["dist_to_track_mi"], m.pvalues["dist_to_track_mi"]
        sigf = "**" if pf < 0.05 else "*" if pf < 0.10 else ""
        print(f"  + {fema_var:<22}: Adj.R²={m.rsquared_adj:.3f}  "
              f"FEMA β={bf:+.3f} (p={pf:.4f}){sigf}   "
              f"dist_to_track β={bd:+.3f} (p={pd_:.4f})")
        summary_rows.append({"dv": dv, "spec": f"+{fema_var}", "N": n,
                              "adjR2": round(m.rsquared_adj, 3),
                              "beta_dist": round(bd, 3), "p_dist": round(pd_, 4),
                              "beta_fema": round(bf, 3), "p_fema": round(pf, 4)})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(EXPOSURE_DIR / "ols_fema_comparison.csv", index=False)
print(f"\nSaved: {EXPOSURE_DIR / 'ols_fema_comparison.csv'}")

## §7. Verdict interpretation

Three possible outcomes:

1. **FEMA strongly significant + dist_to_track loses significance** → FEMA captures the integrated damage that was confounding our other predictors. Use FEMA as the main exposure proxy in the manuscript.
2. **FEMA significant + dist_to_track still significant** → They capture different things. Keep both in the headline model.
3. **FEMA non-significant** → FEMA bundling didn't help; the existing baseline is the best we can do without more granular damage data.

Whichever outcome holds, update [findings.md](../notes/findings.md) §"Affected-region cutoff sensitivity" with the FEMA result and decide accordingly. If outcome 1 or 2, this is a much stronger story than the precipitation result and worth a paragraph in the manuscript.

**Endogeneity reminder for the paper:** be explicit in the methods that FEMA designation is an *integrated damage outcome* used as a graded composite of observed storm impact, not a pre-storm exposure. The interpretation is "places that the federal damage assessment classified as more severely impacted had larger mobility drops, controlling for socioeconomic structure," not "FEMA designation caused the drops."